# 05 — Query ohsome API for all 895 Petrópolis 1 km cells

This production notebook scales the four endpoint patterns validated in Notebook 04 from the deterministic 10-cell sample to all **895** complete 1 km cells intersecting Petrópolis.

It uses conservative batching, one raw checkpoint per endpoint and batch, retry/backoff, strict boundary-ID validation, metadata/version recording, and final integrity checks. Successful batch responses are reusable cache files: when they already exist and pass validation, the request is not repeated.

The four validated metrics are: current road length, current road-element count, unique contributors over the full OSHDB temporal extent, and contributions over the full OSHDB temporal extent.

In [1]:
from pathlib import Path
import json
import math
import time
from datetime import datetime, timezone

import pandas as pd
import geopandas as gpd
import requests

API_BASE = 'https://api.ohsome.org/v1'
ROAD_FILTER = 'highway=* and geometry:line'
BATCH_SIZE = 25
REQUEST_TIMEOUT = 240
MAX_ATTEMPTS = 5
TRANSIENT_STATUS = {429, 500, 502, 503, 504}

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, start.parent, *start.parents]:
        if (candidate / 'notebooks').exists() and (candidate / 'Petropolis').exists():
            return candidate
    raise FileNotFoundError('Could not locate repository root.')

PROJECT_ROOT = find_project_root(Path.cwd())
INPUT_DIR = PROJECT_ROOT / 'data' / 'intermediate' / 'ohsome' / 'Petropolis'
FULL_BOUNDARIES_FILE = INPUT_DIR / 'petropolis_grid_1km_ohsome_boundaries.geojson'
GRID_FILE = PROJECT_ROOT / 'data' / 'processed' / 'Petropolis' / 'petropolis_grid_1km.parquet'
RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'ohsome' / 'Petropolis' / 'production_895_cells'
RESULTS_DIR = PROJECT_ROOT / 'data' / 'processed' / 'Petropolis'
OUTPUT_PARQUET = RESULTS_DIR / 'petropolis_ohsome_895_cells.parquet'
OUTPUT_GEOPARQUET = RESULTS_DIR / 'petropolis_grid_1km_with_ohsome_metrics.parquet'
MANIFEST_FILE = RAW_DIR / 'manifest.json'

for p in (FULL_BOUNDARIES_FILE, GRID_FILE):
    if not p.exists():
        raise FileNotFoundError(f'Required upstream output not found: {p}')
RAW_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

session = requests.Session()
session.headers.update({'User-Agent':'sotm-br-2026-research/0.3 (academic OSM analysis)'})

## 1. Read current ohsome metadata and freeze the query extent

In [2]:
r = session.get(f'{API_BASE}/metadata', timeout=REQUEST_TIMEOUT)
r.raise_for_status()
metadata = r.json()
metadata_file = RAW_DIR / 'ohsome_metadata.json'
metadata_file.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8')

api_version = metadata.get('apiVersion')
temporal = metadata.get('extractRegion', {}).get('temporalExtent', {})
oshdb_start = temporal.get('fromTimestamp')
oshdb_end = temporal.get('toTimestamp')
if not oshdb_start or not oshdb_end:
    raise ValueError('Could not read extractRegion.temporalExtent from /metadata.')

users_time = f'{oshdb_start}/{oshdb_end}'
contributions_time = f'{oshdb_start},{oshdb_end}'
print('API version:', api_version)
print('OSHDB extent:', oshdb_start, '→', oshdb_end)

API version: 1.10.4
OSHDB extent: 2007-10-08T00:00:00Z → 2026-07-27T09:00Z


## 2. Validate all 895 identified boundaries and build deterministic batches

In [3]:
grid = gpd.read_parquet(GRID_FILE)
feature_collection = json.loads(FULL_BOUNDARIES_FILE.read_text(encoding='utf-8'))
features = feature_collection.get('features', [])
feature_by_id = {str(f.get('properties', {}).get('id')): f for f in features}
grid_ids = grid['grid_1km_id'].astype(str).tolist()
boundary_ids = list(feature_by_id)

checks = {
    'grid_rows_895': len(grid) == 895,
    'boundaries_895': len(features) == 895,
    'grid_ids_unique': grid['grid_1km_id'].is_unique,
    'boundary_ids_unique': len(boundary_ids) == len(set(boundary_ids)),
    'ids_match': set(grid_ids) == set(boundary_ids),
}
print(checks)
if not all(checks.values()):
    raise ValueError(f'Full boundary validation failed: {checks}')

ordered_ids = sorted(grid_ids)
batches = [ordered_ids[i:i+BATCH_SIZE] for i in range(0, len(ordered_ids), BATCH_SIZE)]
print(f'Batches: {len(batches)}; batch size: {BATCH_SIZE}; final batch: {len(batches[-1])}')

def batch_feature_collection(ids):
    return {'type':'FeatureCollection', 'features':[feature_by_id[i] for i in ids]}

{'grid_rows_895': True, 'boundaries_895': True, 'grid_ids_unique': True, 'boundary_ids_unique': True, 'ids_match': True}
Batches: 36; batch size: 25; final batch: 20


## 3. Endpoint definitions, raw checkpoints, retry/backoff, and schema validation

In [4]:
ENDPOINTS = {
    'road_length_m': {
        'endpoint':'/elements/length/groupBy/boundary', 'temporal_kind':'snapshot', 'extra':{}
    },
    'road_element_count': {
        'endpoint':'/elements/count/groupBy/boundary', 'temporal_kind':'snapshot', 'extra':{}
    },
    'contributors_count': {
        'endpoint':'/users/count/groupBy/boundary', 'temporal_kind':'interval', 'extra':{'time':users_time}
    },
    'contributions_count': {
        'endpoint':'/contributions/count/groupBy/boundary', 'temporal_kind':'interval', 'extra':{'time':contributions_time}
    },
}

def validate_body(body, expected_ids, temporal_kind):
    if not isinstance(body, dict):
        raise ValueError('Response body is not a JSON object.')
    groups = body.get('groupByResult')
    if not isinstance(groups, list):
        raise ValueError('Missing groupByResult list.')
    returned = [str(g.get('groupByObject')) for g in groups]
    if len(returned) != len(set(returned)):
        raise ValueError('Duplicate groupByObject values in response.')
    if set(returned) != set(expected_ids):
        raise ValueError(f'Boundary mismatch: missing={sorted(set(expected_ids)-set(returned))}, extra={sorted(set(returned)-set(expected_ids))}')
    for g in groups:
        result = g.get('result')
        if not isinstance(result, list) or len(result) != 1:
            raise ValueError(f'Expected one result interval for {g.get("groupByObject")}.')
        item = result[0]
        if 'value' not in item:
            raise ValueError('Result item is missing value.')
        if temporal_kind == 'snapshot' and 'timestamp' not in item:
            raise ValueError('Snapshot result is missing timestamp.')
        if temporal_kind == 'interval' and not {'fromTimestamp','toTimestamp'}.issubset(item):
            raise ValueError('Interval result is missing fromTimestamp/toTimestamp.')
    return True

def checkpoint_path(metric, batch_no):
    return RAW_DIR / metric / f'batch_{batch_no:03d}.json'

def request_or_cache(metric, cfg, batch_no, ids):
    path = checkpoint_path(metric, batch_no)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        cached = json.loads(path.read_text(encoding='utf-8'))
        body = cached.get('response', {}).get('body')
        if cached.get('response', {}).get('status_code') == 200:
            validate_body(body, ids, cfg['temporal_kind'])
            return body, 'cache'

    fc = batch_feature_collection(ids)
    data = {
        'bpolys': json.dumps(fc, ensure_ascii=False, separators=(',',':')),
        'filter': ROAD_FILTER,
        'format': 'json',
        **cfg['extra'],
    }
    url = f"{API_BASE}{cfg['endpoint']}"
    last_error = None
    for attempt in range(1, MAX_ATTEMPTS + 1):
        response = session.post(url, data=data, timeout=REQUEST_TIMEOUT)
        try:
            body = response.json()
        except Exception:
            body = {'raw_text':response.text}
        envelope = {
            'request': {
                'metric':metric, 'endpoint':cfg['endpoint'], 'batch_no':batch_no,
                'ids':ids, 'feature_count':len(ids),
                'filter':ROAD_FILTER, 'extra':cfg['extra'],
            },
            'response': {
                'status_code':response.status_code,
                'content_type':response.headers.get('content-type',''),
                'body':body,
            },
            'recorded_at_utc':datetime.now(timezone.utc).isoformat(),
        }
        if response.ok:
            validate_body(body, ids, cfg['temporal_kind'])
            path.write_text(json.dumps(envelope, indent=2, ensure_ascii=False), encoding='utf-8')
            return body, 'request'
        error_path = path.with_suffix('.error.json')
        error_path.write_text(json.dumps(envelope, indent=2, ensure_ascii=False), encoding='utf-8')
        last_error = RuntimeError(f"{metric} batch {batch_no} HTTP {response.status_code}: {json.dumps(body, ensure_ascii=False)[:1000]}")
        if response.status_code not in TRANSIENT_STATUS or attempt == MAX_ATTEMPTS:
            raise last_error
        retry_after = response.headers.get('Retry-After')
        delay = float(retry_after) if retry_after and retry_after.isdigit() else min(2 ** attempt, 30)
        time.sleep(delay)
    raise last_error

def parse_body(body, metric, temporal_kind):
    rows = []
    for group in body['groupByResult']:
        gid = str(group['groupByObject'])
        item = group['result'][0]
        row = {'grid_1km_id':gid, metric:float(item['value'])}
        if temporal_kind == 'snapshot':
            row[f'{metric}_timestamp'] = item['timestamp']
        else:
            row[f'{metric}_from'] = item['fromTimestamp']
            row[f'{metric}_to'] = item['toTimestamp']
        rows.append(row)
    return pd.DataFrame(rows)

## 4. Execute or resume all batches

In [5]:
metric_frames = {}
run_stats = []
for metric, cfg in ENDPOINTS.items():
    parts = []
    for batch_no, ids in enumerate(batches, start=1):
        body, source = request_or_cache(metric, cfg, batch_no, ids)
        parts.append(parse_body(body, metric, cfg['temporal_kind']))
        run_stats.append({'metric':metric, 'batch_no':batch_no, 'cells':len(ids), 'source':source})
        print(f'{metric}: batch {batch_no:02d}/{len(batches)} — {len(ids)} cells — {source}')
    metric_frames[metric] = pd.concat(parts, ignore_index=True)

run_stats_df = pd.DataFrame(run_stats)
display(run_stats_df.groupby(['metric','source']).agg(batches=('batch_no','count'), cells=('cells','sum')).reset_index())

road_length_m: batch 01/36 — 25 cells — request


road_length_m: batch 02/36 — 25 cells — request


road_length_m: batch 03/36 — 25 cells — request


road_length_m: batch 04/36 — 25 cells — request


road_length_m: batch 05/36 — 25 cells — request


road_length_m: batch 06/36 — 25 cells — request


... (135 outputs left out)

contributions_count: batch 34/36 — 25 cells — request


contributions_count: batch 35/36 — 25 cells — request


contributions_count: batch 36/36 — 20 cells — request


,metric,source,batches,cells
0,contributions_count,request,36,895
1,contributors_count,request,36,895
2,road_element_count,request,36,895
3,road_length_m,request,36,895


## 5. Consolidate metrics and validate complete 895-cell coverage

In [6]:
result = pd.DataFrame({'grid_1km_id':ordered_ids})
for metric, frame in metric_frames.items():
    if len(frame) != 895 or not frame['grid_1km_id'].is_unique:
        raise ValueError(f'{metric}: invalid parsed coverage: rows={len(frame)}, unique={frame["grid_1km_id"].nunique()}')
    result = result.merge(frame, on='grid_1km_id', how='left', validate='one_to_one')

metric_cols = list(ENDPOINTS)
coverage = pd.DataFrame({
    'metric':metric_cols,
    'non_null':[int(result[c].notna().sum()) for c in metric_cols],
    'zeros':[int(result[c].eq(0).sum()) for c in metric_cols],
    'negative':[int(result[c].lt(0).sum()) for c in metric_cols],
})
coverage['complete_895'] = coverage['non_null'].eq(895)
display(coverage)

final_checks = {
    'rows_895':len(result)==895,
    'ids_unique':result['grid_1km_id'].is_unique,
    'ids_match_grid':set(result['grid_1km_id'])==set(grid_ids),
    'all_metrics_complete':all(result[c].notna().all() for c in metric_cols),
    'all_metrics_nonnegative':all(result[c].ge(0).all() for c in metric_cols),
}
print(final_checks)
if not all(final_checks.values()):
    raise ValueError(f'Production integrity validation failed: {final_checks}')

result['road_length_km'] = result['road_length_m'] / 1000.0
area_lookup = grid.set_index(grid['grid_1km_id'].astype(str))['cell_area_m2']
result['cell_area_m2'] = result['grid_1km_id'].map(area_lookup)
result['road_density_km_per_km2'] = result['road_length_km'] / (result['cell_area_m2']/1_000_000.0)
result['ohsome_api_version'] = api_version
result['oshdb_from'] = oshdb_start
result['oshdb_to'] = oshdb_end
result['road_filter'] = ROAD_FILTER
result['query_run_utc'] = datetime.now(timezone.utc).isoformat()
result.to_parquet(OUTPUT_PARQUET, index=False)

geo = grid.copy()
geo['grid_1km_id'] = geo['grid_1km_id'].astype(str)
geo = geo.merge(result.drop(columns=['cell_area_m2']), on='grid_1km_id', how='left', validate='one_to_one')
geo.to_parquet(OUTPUT_GEOPARQUET, index=False)

manifest = {
    'created_at_utc':datetime.now(timezone.utc).isoformat(),
    'api_version':api_version, 'oshdb_from':oshdb_start, 'oshdb_to':oshdb_end,
    'road_filter':ROAD_FILTER, 'batch_size':BATCH_SIZE, 'batch_count':len(batches),
    'cells':895, 'coverage':coverage.to_dict(orient='records'),
    'cache_stats':run_stats_df.groupby(['metric','source']).size().rename('batches').reset_index().to_dict(orient='records'),
}
MANIFEST_FILE.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')
print('PASS — complete 895-cell production query validated.')
print('Saved:', OUTPUT_PARQUET)
print('Saved:', OUTPUT_GEOPARQUET)
print('Manifest:', MANIFEST_FILE)

,metric,non_null,zeros,negative,complete_895
0,road_length_m,895,114,0,True
1,road_element_count,895,114,0,True
2,contributors_count,895,100,0,True
3,contributions_count,895,100,0,True


{'rows_895': True, 'ids_unique': True, 'ids_match_grid': True, 'all_metrics_complete': True, 'all_metrics_nonnegative': True}
PASS — complete 895-cell production query validated.
Saved: /home/runner/work/sotm-br-2026/sotm-br-2026/data/processed/Petropolis/petropolis_ohsome_895_cells.parquet
Saved: /home/runner/work/sotm-br-2026/sotm-br-2026/data/processed/Petropolis/petropolis_grid_1km_with_ohsome_metrics.parquet
Manifest: /home/runner/work/sotm-br-2026/sotm-br-2026/data/raw/ohsome/Petropolis/production_895_cells/manifest.json
